# UST-Fuse Cyber Range — цифровий двійник (Colab)

**Бюджетний гібридний навчально-науковий кіберполігон UST-Fuse.**
Цей ноутбук запускає цифровий двійник полігону виявлення/супроводження БПЛА,
проганяє сценарії, будує максимум візуалізацій і формує звіти як на польових
іспитах. Оформлено як програмний проєкт для Zenodo.

*A budget, virtual-first digital twin of the UST-Fuse multi-sensor UAV
detection/tracking range — scenarios, rich visualisations, and field-trial
reports. Packaged as a Zenodo software record.*

---
**Як користуватись / How to use:** `Runtime → Run all`. Нічого екзотичного не
встановлюється (numpy / scipy / matplotlib / pandas / pyyaml).


## 0. Встановлення та імпорт / Setup
У Colab розкоментуйте рядок `git clone`, щоб отримати пакет із репозиторію.


In [ ]:
# --- Colab: отримати пакет із репозиторію (розкоментуйте) ---
# !git clone https://github.com/omega2417/bnt.git
# %cd bnt/ust_fuse_cyber_range

!pip -q install numpy scipy matplotlib pandas pyyaml >/dev/null 2>&1

import os, sys
# зробити пакет доступним із каталогу проєкту (локально або після clone)
for cand in ['src', 'ust_fuse_cyber_range/src', '../src']:
    if os.path.isdir(cand):
        sys.path.insert(0, os.path.abspath(cand)); break

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ust_fuse as uf
from ust_fuse.viz import (set_style, mission_dashboard, figure_pack,
    plot_topdown, plot_sensor_coverage, plot_error_over_time, plot_ospa_over_time,
    plot_roc, plot_reliability_diagram, plot_selective_risk, plot_clock_desync,
    plot_metric_comparison, plot_paired_forest, plot_campaign_box,
    plot_scenario_heatmap, plot_fault_timeline, plot_track_confidence,
    plot_trajectories_3d, plot_detections)
from ust_fuse.campaign import Campaign
from ust_fuse.domain import randomize_scenario
from ust_fuse.report import write_report, build_markdown_report
set_style()
print('UST-Fuse version', uf.__version__)

## 1. Полігон і бібліотека сценаріїв / The range & scenario library
Бюджетний MVP-набір сенсорів: радар (партнерський/орендований), EO-IR камера,
пасивний SDR та акустичний масив. 12 сценаріїв покривають ЛР-1…ЛР-10 + red-team.


In [ ]:
from ust_fuse.config import default_range
rng_cfg = default_range()
print('Sensors:')
for s in rng_cfg.sensors:
    print(f'  {s.sensor_id:6s} {s.sensor_type:9s} range={s.max_range:5.0f}m  '
          f'{"RANGE" if s.provides_range else "bearing"}  classify={s.can_classify}')
print()
for sid in uf.list_scenarios():
    scn = uf.build_scenario(sid)
    print(f'  {sid:28s} [{scn.lab or "—":6s}] {scn.title}')
plot_sensor_coverage(rng_cfg); plt.show()

## 2. Один експеримент / One experiment
Проганяємо сценарій відмови радара (ЛР-5). Обидва режими злиття — Reference та
Full UST-Fuse — працюють на **однакових незмінюваних RAW-даних**.


In [ ]:
res = uf.run('S04_sensor_dropout', seed=7)
print('Experiment:', res.manifest.experiment_id, '| seed', res.config.seed)
print('RAW detections:', res.raw.stats['n_detections'],
      '| clutter %.1f%%' % (100*res.raw.stats['clutter_fraction']))
summary = pd.DataFrame(res.summary_table())
cols = ['mode','rmse_pos','ospa_mean','mota','id_switches','n_false_tracks',
        'track_completeness','ece','brier']
summary[[c for c in cols if c in summary.columns]].round(3)

### 2.1 Зведена панель місії / Mission dashboard
Одна панель — уся місія: вид зверху, похибка та OSPA у часі, ROC, калібрування, Pd.


In [ ]:
mission_dashboard(res); plt.show()

### 2.2 Окремі візуалізації / Individual visualisations


In [ ]:
plot_topdown(res, mode='ust_fuse'); plt.show()
plot_trajectories_3d(res, mode='ust_fuse'); plt.show()
plot_error_over_time(res); plt.show()
plot_ospa_over_time(res); plt.show()

In [ ]:
plot_reliability_diagram(res, mode='ust_fuse'); plt.show()
plot_selective_risk(res); plt.show()
plot_clock_desync(res); plt.show()
plot_fault_timeline(res); plt.show()
plot_track_confidence(res, mode='ust_fuse'); plt.show()
plot_metric_comparison(res); plt.show()

## 3. Калібрування часу (ЛР-1) / Time calibration
Оцінені зсув і дрейф годинників кожного сенсора; Full UST-Fuse їх коригує.


In [ ]:
res_clock = uf.run('S02_time_calibration', seed=3)
est = res_clock.manifest.extra['clock_estimates']
pd.DataFrame([{'sensor':k, **{kk:round(vv,3) for kk,vv in v.items() if kk!='sensor_id'}}
              for k,v in est.items()])

## 4. Domain randomization (ЛР-6) / один політ → багато варіантів
Один базовий сценарій розмножуємо у сім'ю місій з різною погодою/шумом/клатером.


In [ ]:
from ust_fuse.rng import RNGHub
base = uf.build_scenario('S01_baseline_clear')
hub = RNGHub(2026)
variants = [randomize_scenario(base, hub, i) for i in range(6)]
for v in variants:
    print(f'  {v.scenario_id:24s} weather={v.weather:6s} '
          f'clutter={v.clutter_scale:.2f} noise={v.noise_scale:.2f}')

## 5. Кампанія та парний аналіз (ЛР-3, ЛР-9) / Paired campaign
Пілотна кампанія: багато місій, парне порівняння Reference vs UST-Fuse з
ефект-сайзом (Cohen's d), 95% ДІ (bootstrap) та аналізом потужності.
(У Colab збільшіть `n_missions` до 20–30.)


In [ ]:
camp = Campaign('S04_sensor_dropout', n_missions=12).run(verbose=True)
tbl = camp.paired_table()
show = tbl[['metric','mean_a','mean_b','mean_diff','ci_low','ci_high',
            'cohens_d','power','better']].copy()
show['better'] = show['better'].map({'A':'Reference','B':'UST-Fuse'})
show.round(3)

In [ ]:
plot_paired_forest(camp); plt.show()
plot_campaign_box(camp, 'track_completeness'); plt.show()
plot_campaign_box(camp, 'ece'); plt.show()

## 6. Матриця сценаріїв / Scenario suite heatmap
Проганяємо всі сценарії й будуємо теплову карту метрики за сценаріями × режимами.


In [ ]:
rows = []
for sid in uf.list_scenarios():
    r = uf.run(sid, seed=11)
    for m in r.summary_table():
        m = dict(m); m['scn'] = sid; rows.append(m)
suite = pd.DataFrame(rows)
plot_scenario_heatmap(suite, 'track_completeness'); plt.show()
plot_scenario_heatmap(suite, 'mota'); plt.show()
suite[['scn','mode','rmse_pos','mota','ospa_mean','track_completeness','ece']].round(2)

## 7. Звіт польових випробувань / Field-trial report
Формуємо повний двомовний звіт (Markdown + HTML) з рисунками, таблицями,
провенансом і KPI-скорингом — як на польових іспитах (ЛР-10).


In [ ]:
out_dir = 'out/colab_s04'
figs = figure_pack(res, out_dir)
paths = write_report(res, out_dir, figures=figs, campaign=camp)
print('Report written:')
print(' ', paths['markdown'])
print(' ', paths['html'])
# у Colab: завантажити HTML-звіт
try:
    from google.colab import files  # type: ignore
    files.download(paths['html'])
except Exception:
    print('(поза Colab — відкрийте файл локально)')

## 8. Відтворюваність / Reproducibility (ЛР-8)
Кожен результат відтворюється з `seed` та config hash; маніфест фіксує версії.


In [ ]:
import json
print(res.manifest.to_json())

---
### Висновок / Conclusion
Цифровий двійник відтворює польову логіку полігону UST-Fuse у віртуальному
середовищі: єдиний час, незмінювані RAW-дані, однакові вхідні потоки для парного
порівняння, статистику з ефект-сайзом і ДІ, багато сценаріїв, максимум
візуалізацій та автоматичні звіти — з повною простежуваністю від RAW до рисунків.

*Cite via `CITATION.cff`; archive via `.zenodo.json`.*
